In [ ]:
#@title Run this cell to set up and launch the AI Image Studio

# Step 1: Install all necessary libraries quietly
!pip install -q gradio
!pip install -q torch torchvision torchaudio
!pip install -q diffusers transformers accelerate
!pip install -q opencv-python-headless
!pip install -q Pillow

In [ ]:
# Step 2: Import Libraries
import gradio as gr
import torch
import numpy as np
from PIL import Image
import cv2
import os # Though not heavily used in the final functions, good practice to keep if potentially needed
from diffusers import StableDiffusionPipeline
import warnings

# Suppress specific warnings if they become noisy, though usually it's better to address the root cause
# warnings.filterwarnings("ignore", category=FutureWarning)

print("Libraries installed and imported.")


Libraries installed and imported.


In [ ]:
# --- Step 3: Configuration & Model Loading ---

# Check for GPU availability (standard practice in Colab)
if torch.cuda.is_available():
    device = "cuda"
    # Use float16 for memory efficiency on GPU
    torch_dtype = torch.float16
    print("CUDA (GPU) is available. Using GPU.")
else:
    device = "cpu"
    torch_dtype = torch.float32 # float32 is more standard for CPU
    print("CUDA not available. Using CPU (Stable Diffusion generation will be very slow).")

# --- Global Variables ---
# Load Stable Diffusion Model (only once when the cell runs)
# Using Stable Diffusion v1.5 as a standard base.
MODEL_ID = "runwayml/stable-diffusion-v1-5"
print(f"Loading Stable Diffusion model: {MODEL_ID}...")
print("This may take a few minutes, especially the first time it downloads...")

try:
    pipe = StableDiffusionPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch_dtype
    )
    pipe = pipe.to(device)

    print("Stable Diffusion model loaded successfully.")
    SD_LOADED = True
except ImportError as e:
     print(f"ImportError loading model: {e}. Did you install all requirements? Particularly 'transformers'?")
     print("Image generation will be disabled.")
     pipe = None
     SD_LOADED = False
except Exception as e:
    print(f"An error occurred loading the Stable Diffusion model: {e}")
    print("Image generation will be disabled.")
    pipe = None
    SD_LOADED = False

# Placeholder for style images (paths are illustrative, not functional unless you upload files)
# For the 'Pencil Sketch' we implement it directly with OpenCV, so no path needed.
STYLE_IMAGES = {
    "Van Gogh": "path/to/van_gogh.jpg", # Replace with an actual path if you upload a style image
    "Anime": "path/to/anime_style.jpg",   # Replace with an actual path if you upload a style image
    "Pencil Sketch": None # This style is generated using OpenCV
}

CUDA (GPU) is available. Using GPU.
Loading Stable Diffusion model: runwayml/stable-diffusion-v1-5...
This may take a few minutes, especially the first time it downloads...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

An error occurred loading the Stable Diffusion model: Cannot copy out of meta tensor; no data! Please use torch.nn.Module.to_empty() instead of torch.nn.Module.to() when moving module from meta to a different device.
Image generation will be disabled.


In [ ]:
# --- Step 4: Core Functions ---

def generate_image_sd(prompt: str, guidance_scale: float = 7.5, num_steps: int = 30) -> Image.Image:
    """
    Generates an image from a text prompt using Stable Diffusion.
    Args:
        prompt: The text prompt to generate the image from.
        guidance_scale: Controls how much the generation follows the prompt (higher means stricter).
        num_steps: Number of diffusion steps (higher means potentially better quality but slower).
                   Reduced default to 30 for faster Colab execution.
    Returns:
        A PIL Image object or raises a Gradio error if generation failed.
    """
    if not SD_LOADED or pipe is None:
        raise gr.Error("Stable Diffusion model is not loaded. Cannot generate image.")

    print(f"Generating image for prompt: '{prompt}' with guidance {guidance_scale} and {num_steps} steps on device {device}.")

    # If using CPU, explicitly limit steps to prevent excessive runtimes
    if device == 'cpu' and num_steps > 20:
        print("Warning: Running on CPU. Limiting steps to 20 for reasonable performance.")
        num_steps = 20

    try:
        # Ensure inference_mode for efficiency (no gradient calculations)
        with torch.inference_mode():
             # Generate the image
             # Consider adding negative_prompt for more control if desired
             result = pipe(
                 prompt,
                 guidance_scale=guidance_scale,
                 num_inference_steps=int(num_steps) # Ensure steps is int
             )
             # Access the generated image
             image = result.images[0]

        print("Image generated successfully.")
        # Clear CUDA cache if memory is tight (optional, might slightly slow down back-to-back runs)
        # if device == 'cuda':
        #    torch.cuda.empty_cache()
        return image

    except torch.cuda.OutOfMemoryError:
         # Attempt to clear cache and raise error
         if device == 'cuda':
            torch.cuda.empty_cache()
         raise gr.Error("GPU out of memory (OOM). Try reducing the number of inference steps or use a Colab runtime with more VRAM. If the problem persists, the prompt might be too complex for available memory.")
    except Exception as e:
        print(f"Error during image generation: {e}")
        # Clear cache on any exception just in case it helps
        if device == 'cuda':
            torch.cuda.empty_cache()
        raise gr.Error(f"Failed to generate image: {e}")

def apply_style_transfer_simple(content_image_np: np.ndarray, style_name: str) -> np.ndarray:
    """
    Applies a *simplified* style transfer or effect using OpenCV.
    NOTE: This is a placeholder/demonstration. Real AI style transfer uses dedicated models.
    Args:
        content_image_np: The input image as a NumPy array (BGR format from Gradio).
        style_name: The name of the style to apply.
    Returns:
        The modified image as a NumPy array (BGR format).
    """
    print(f"Applying style (simplified OpenCV effect): {style_name}")
    if content_image_np is None:
        # Return a blank image or raise error if no input
        # raise gr.Error("Please upload an image first.")
        # Returning placeholder might be friendlier
        return np.zeros((256, 256, 3), dtype=np.uint8) # Return small black square

    # --- Simple OpenCV Implementations or Placeholders ---
    if style_name == "Van Gogh":
        # Placeholder: Increase saturation and add some swirling effect (e.g., via blur/sharpen)
        try:
            hsv = cv2.cvtColor(content_image_np, cv2.COLOR_BGR2HSV)
            # Increase saturation carefully, clip ensures it stays in valid range
            hsv[:, :, 1] = np.clip(hsv[:, :, 1] * 1.5, 0, 255)
            styled_image = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
            # Apply OpenCV's stylization filter (can mimic painterly effects)
            styled_image = cv2.stylization(styled_image, sigma_s=60, sigma_r=0.6)
            print("Applied 'Van Gogh' (simplified: increased saturation + CV2 stylization filter).")
        except Exception as e:
            print(f"Error applying Van Gogh style: {e}")
            styled_image = content_image_np # Return original on error

    elif style_name == "Anime":
        # Placeholder: Use bilateral filter for smoothing + edge detection overlay?
        # Or just a cartoon effect using stylization
        try:
            styled_image = cv2.stylization(content_image_np, sigma_s=150, sigma_r=0.25)
            print("Applied 'Anime' (simplified: cartoon effect using cv2.stylization).")
        except Exception as e:
            print(f"Error applying Anime style: {e}")
            styled_image = content_image_np

    elif style_name == "Pencil Sketch":
        try:
            # Convert to grayscale
            gray_image = cv2.cvtColor(content_image_np, cv2.COLOR_BGR2GRAY)
            # Invert the grayscale image
            inverted_gray = 255 - gray_image
            # Apply Gaussian blur to the inverted image - kernel size affects sketchiness
            blurred = cv2.GaussianBlur(inverted_gray, (21, 21), 0)
            # Invert the blurred image
            inverted_blurred = 255 - blurred
            # Create the pencil sketch effect by dividing the grayscale image by the inverted blurred image
            # Add a small epsilon to denominator to avoid division by zero
            styled_image_gray = cv2.divide(gray_image, inverted_blurred + 1e-5, scale=256.0)
            # Convert back to BGR for Gradio display (will still look grayscale)
            styled_image = cv2.cvtColor(styled_image_gray, cv2.COLOR_GRAY2BGR)
            print("Applied 'Pencil Sketch' effect using OpenCV.")
        except Exception as e:
            print(f"Error applying Pencil Sketch style: {e}")
            styled_image = content_image_np

    else:
        print(f"Style '{style_name}' not implemented with a simple version. Returning original.")
        styled_image = content_image_np # Return original if style not found

    return styled_image

def colorize_image_placeholder(grayscale_image_np: np.ndarray) -> np.ndarray:
    """
    Placeholder for AI colorization. Converts input to grayscale and back to BGR.
    NOTE: Real AI colorization requires a dedicated model.
    Args:
        grayscale_image_np: Input image (Gradio might provide color, we ensure grayscale).
    Returns:
        A BGR image (simulating colorization output, but will be grayscale).
    """
    print("Applying Colorization (placeholder - converting to grayscale and back).")
    if grayscale_image_np is None:
        # raise gr.Error("Please upload an image first.")
        return np.zeros((256, 256, 3), dtype=np.uint8)

    try:
        # Ensure the input is grayscale
        if len(grayscale_image_np.shape) == 3 and grayscale_image_np.shape[2] == 3:
            # Input was color, convert it to grayscale first
            grayscale_image = cv2.cvtColor(grayscale_image_np, cv2.COLOR_BGR2GRAY)
        elif len(grayscale_image_np.shape) == 2:
             # Input was already grayscale
             grayscale_image = grayscale_image_np
        else:
            print("Warning: Unexpected image format for colorization placeholder.")
            return grayscale_image_np # Return original if format is weird

        # --- Placeholder implementation ---
        # Just convert the grayscale image back to BGR format for display consistency
        colorized_image_bgr = cv2.cvtColor(grayscale_image, cv2.COLOR_GRAY2BGR)
        print("Applied placeholder colorization (image will appear grayscale).")
    except Exception as e:
        print(f"Error applying colorization placeholder: {e}")
        colorized_image_bgr = grayscale_image_np # Return original on error

    return colorized_image_bgr

def upscale_image_simple(image_np: np.ndarray, scale_factor: int = 2) -> np.ndarray:
    """
    Upscales an image using simple OpenCV resizing (interpolation).
    NOTE: Real AI upscaling (e.g., ESRGAN, Real-ESRGAN) provides much better results.
    Args:
        image_np: Input image as NumPy array (BGR).
        scale_factor: Factor by which to upscale (e.g., 2 for 2x).
    Returns:
        Upscaled image as NumPy array (BGR).
    """
    print(f"Upscaling image by {scale_factor}x (simple interpolation using Lanczos4).")
    if image_np is None:
        # raise gr.Error("Please upload an image first.")
        return np.zeros((256, 256, 3), dtype=np.uint8)

    try:
        height, width = image_np.shape[:2]
        new_width = int(width * scale_factor)
        new_height = int(height * scale_factor)

        # Use cv2.resize with a good quality interpolation method like Lanczos4 or Cubic
        # Avoid INTER_NEAREST for upscaling photos.
        upscaled_image = cv2.resize(image_np, (new_width, new_height), interpolation=cv2.INTER_LANCZOS4)
        print(f"Image resized to {new_width}x{new_height}.")
    except Exception as e:
        print(f"Error during simple upscaling: {e}")
        upscaled_image = image_np # Return original on error

    return upscaled_image

def apply_filter_cv(image_np: np.ndarray, filter_type: str) -> np.ndarray:
    """
    Applies a basic image filter using OpenCV.
    Args:
        image_np: Input image as NumPy array (BGR).
        filter_type: Name of the filter to apply ('Blur', 'Sharpen', 'Grayscale', 'Sepia').
    Returns:
        Filtered image as NumPy array (BGR).
    """
    print(f"Applying OpenCV filter: {filter_type}")
    if image_np is None:
        # raise gr.Error("Please upload an image first.")
        return np.zeros((256, 256, 3), dtype=np.uint8)

    try:
        if filter_type == "Blur":
            # Apply Gaussian Blur - kernel size affects strength (must be odd)
            filtered_image = cv2.GaussianBlur(image_np, (15, 15), 0)
            print("Applied Gaussian Blur.")
        elif filter_type == "Sharpen":
            # Apply sharpening using a kernel
            # Kernel enhances differences between center pixel and neighbors
            kernel = np.array([[-1, -1, -1],
                               [-1,  9, -1],
                               [-1, -1, -1]])
            filtered_image = cv2.filter2D(image_np, -1, kernel)
            print("Applied Sharpening filter.")
        elif filter_type == "Grayscale":
            # Convert to grayscale and back to BGR for consistent 3-channel output
            gray = cv2.cvtColor(image_np, cv2.COLOR_BGR2GRAY)
            filtered_image = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
            print("Applied Grayscale filter.")
        elif filter_type == "Sepia":
             # Apply sepia filter using a standard transformation matrix
            kernel = np.array([[0.272, 0.534, 0.131], # BGR coefficients
                               [0.349, 0.686, 0.168],
                               [0.393, 0.769, 0.189]])
            # Apply the kernel - requires float conversion & clipping
            img_float = image_np.astype(np.float32) / 255.0 # Normalize to [0, 1]
            # Apply kernel. Note: cv2.transform expects kernel transposed vs standard matrix math
            sepia_img_float = cv2.transform(img_float, kernel.T)
            # Clip values to ensure they stay within [0, 1] range
            sepia_img_float = np.clip(sepia_img_float, 0, 1)
            # Convert back to uint8 for display
            filtered_image = (sepia_img_float * 255.0).astype(np.uint8)
            print("Applied Sepia filter.")
        else:
            print(f"Filter type '{filter_type}' not recognized. Returning original image.")
            filtered_image = image_np
    except Exception as e:
        print(f"Error applying filter '{filter_type}': {e}")
        filtered_image = image_np # Return original on error

    return filtered_image


In [ ]:
# --- Step 5: Gradio Interface ---

print("Setting up Gradio interface...")

# Use a Soft theme for Gradio
theme = gr.themes.Soft(primary_hue=gr.themes.colors.blue, secondary_hue=gr.themes.colors.sky)

with gr.Blocks(theme=theme) as app:
    gr.Markdown("# 🎨 AI Image Generation & Modification Studio (Colab Version)")
    gr.Markdown("Generate new images from text using Stable Diffusion or upload and modify your existing images.")

    with gr.Tabs():
        # --- Generation Tab ---
        with gr.TabItem("✨ Generate Image (Text-to-Image)"):
            gr.Markdown("### Using Stable Diffusion v1.5")
            if not SD_LOADED:
                 gr.Markdown("⚠️ **Stable Diffusion model failed to load. Generation is disabled.** Check console output above for errors.")

            with gr.Row():
                with gr.Column(scale=3): # Make prompt area wider
                    prompt_input = gr.Textbox(
                        label="Enter your prompt:",
                        placeholder="e.g., A photo of an astronaut riding a horse on the moon",
                        lines=3 # Allow multi-line input
                    )
                    # Consider adding a Negative Prompt input for more advanced control
                    # negative_prompt_input = gr.Textbox(label="Negative Prompt (optional):", placeholder="e.g., blurry, low quality, text, watermark", lines=2)

                    with gr.Row(): # Put sliders side-by-side
                         guidance_slider = gr.Slider(minimum=1.0, maximum=20.0, value=7.5, step=0.5, label="Guidance Scale", info="How strictly the image should follow the prompt (higher=stricter).")
                         steps_slider = gr.Slider(minimum=10, maximum=75, value=30, step=1, label="Inference Steps", info="More steps potentially increase quality but take longer (Recommend ~20-40).")

                    generate_button = gr.Button("Generate Image", variant="primary", interactive=SD_LOADED) # Button is disabled if model failed load

                with gr.Column(scale=2): # Area for the output image
                    generated_output = gr.Image(
                        label="Generated Image",
                        type="pil" # Output as PIL for easy saving by user
                    )
                    # Add a download button perhaps? Gradio Image output usually has one by default.

            # --- Wire the Generation Button ---
            generate_button.click(
                fn=generate_image_sd,
                # If adding negative prompt: inputs=[prompt_input, negative_prompt_input, guidance_slider, steps_slider],
                inputs=[prompt_input, guidance_slider, steps_slider],
                outputs=generated_output,
                api_name="generate_image" # Allows calling via API if needed
            )

            # --- Examples ---
            gr.Examples(
                label="Example Prompts",
                examples=[
                    ["A cinematic photo of a corgi wearing sunglasses on a skateboard, dramatic lighting", 7.5, 35],
                    ["Impressionist oil painting of a bustling market scene in Marrakesh", 6.0, 30],
                    ["A watercolor illustration of a magical forest with glowing mushrooms", 8.0, 40],
                    ["Logo for a cozy bookstore called 'The Wandering Quill', clean vector style", 9.0, 25],
                ],
                inputs=[prompt_input, guidance_slider, steps_slider],
                outputs=generated_output,
                fn=generate_image_sd,
                cache_examples=False # Don't cache examples in Colab as environment can be transient
            )


        # --- Modification Tab ---
        with gr.TabItem("🖼️ Modify Existing Image"):
            gr.Markdown("Upload an image and apply various modifications.")

            with gr.Row():
                # Input Image - Use NumPy format for OpenCV compatibility
                input_image_modify = gr.Image(label="Upload Image for Modification", type="numpy", height=300)
                # Output Image - Display result also as NumPy
                modified_output = gr.Image(label="Modified Image", type="numpy", height=300)

            with gr.Tabs(): # Nested tabs for different modification types
                 # --- Style Transfer Sub-Tab ---
                with gr.TabItem("🎨 Style Transfer (Simplified)"):
                     gr.Markdown("Apply a simplified artistic style using OpenCV filters. (Note: Not true AI style transfer.)")
                     style_choice = gr.Dropdown(list(STYLE_IMAGES.keys()), label="Choose Style Effect", value="Pencil Sketch")
                     style_button = gr.Button("Apply Style Effect", variant="secondary")
                     style_button.click(
                         fn=apply_style_transfer_simple,
                         inputs=[input_image_modify, style_choice],
                         outputs=modified_output,
                         api_name="apply_style_simple"
                     )
                     gr.Markdown("*(Advanced: True AI style transfer requires specific models, e.g., from TensorFlow Hub or PyTorch Hub.)*")


                 # --- Colorization Sub-Tab ---
                with gr.TabItem("🌈 Colorization (Placeholder)"):
                     gr.Markdown("Simulates the input for a colorization model. (Note: This version only converts to grayscale and back; it does not actually add color.)")
                     colorize_button = gr.Button("Run Colorize Placeholder", variant="secondary")
                     colorize_button.click(
                         fn=colorize_image_placeholder,
                         inputs=[input_image_modify],
                         outputs=modified_output,
                         api_name="colorize_placeholder"
                     )
                     gr.Markdown("*(Advanced: Real AI colorization uses models like those by Zhang et al. or DeOldify.)*")


                 # --- Upscaling Sub-Tab ---
                with gr.TabItem("🔍 Upscaling (Simple)"):
                    gr.Markdown("Increase image resolution using standard interpolation. (Note: Not AI super-resolution.)")
                    scale_factor_slider = gr.Slider(minimum=2, maximum=4, value=2, step=1, label="Upscale Factor (2x or 4x)")
                    upscale_button = gr.Button("Upscale with Interpolation", variant="secondary")
                    upscale_button.click(
                        fn=upscale_image_simple,
                        inputs=[input_image_modify, scale_factor_slider],
                        outputs=modified_output,
                        api_name="upscale_simple"
                    )
                    gr.Markdown("*(Advanced: Libraries like Real-ESRGAN provide significantly better AI-based upscaling.)*")

                 # --- Filters Sub-Tab ---
                with gr.TabItem("✨ Filters (OpenCV)"):
                     gr.Markdown("Apply standard image filters using OpenCV.")
                     filter_choice = gr.Dropdown(["Blur", "Sharpen", "Grayscale", "Sepia"], label="Choose Filter", value="Blur")
                     filter_button = gr.Button("Apply Filter", variant="secondary")
                     filter_button.click(
                         fn=apply_filter_cv,
                         inputs=[input_image_modify, filter_choice],
                         outputs=modified_output,
                         api_name="apply_filter"
                     )


    # --- Learning & Ethics Section ---
    with gr.Accordion("📚 Learning Objectives & Ethical Considerations", open=False): # Start closed
        gr.Markdown(
            """
            ### Core Concepts Learned:
            * **Text-to-Image Generation:** How models like Stable Diffusion (a type of Diffusion Model) turn text prompts into images by starting with noise and progressively refining it based on the text guidance.
            * **Diffusion Models:** The basic idea of adding noise and then learning to reverse the process to generate data. Parameters like *Guidance Scale* control prompt adherence, and *Steps* control the refinement process.
            * **Image Processing:** Basic techniques using OpenCV like blurring, sharpening, color space changes (Grayscale, Sepia), and resizing.
            * **AI Image Modification (Concepts):** Understanding the *goal* of Style Transfer, Colorization, and AI Upscaling (Super-Resolution), even if simplified versions are implemented here.
            * **Using Pre-trained Models:** Leveraging powerful models (like Stable Diffusion from Hugging Face) without needing to train them from scratch.
            * **Building Interfaces:** Using Gradio to quickly create interactive web UIs for machine learning models, especially useful in Colab.

            ### Ethical Considerations & Responsible Use:
            * **Copyright & Style:** Be mindful of generating images in the distinct, recognizable style of living artists without permission. Training data copyrights are also a complex area.
            * **Misinformation:** AI image generation can create convincing fakes ("deepfakes"). Use this technology responsibly and ethically. Avoid creating or spreading misleading content.
            * **Bias:** AI models can inherit and amplify biases present in their training data. Be aware that generated images might reflect societal stereotypes.
            * **Content Safety:** The built-in safety checker was disabled here for simplicity. In real applications, consider the risks of generating potentially harmful, explicit, or biased content and implement appropriate safeguards.
            * **Resource Consumption:** Training and running large AI models consumes significant computational resources and energy. Be mindful of this environmental aspect.
            """
        )


Setting up Gradio interface...


In [ ]:
# --- Step 6: Launch the App ---
print("Gradio interface setup complete. Launching the app...")
print("Look for a public URL (like 'https://....gradio.live') in the output below to access the UI.")

# Launch the Gradio app.
# `share=True` was the old way to get a public link in Colab, but it's often automatic now.
# `debug=True` provides helpful error messages in the Colab output.
# `inline=False` can sometimes help if the UI doesn't render correctly within the Colab cell.
app.launch(debug=True)

Gradio interface setup complete. Launching the app...
Look for a public URL (like 'https://....gradio.live') in the output below to access the UI.
It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4f570d59d5c86586fe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4f570d59d5c86586fe.gradio.live
